# 02 - Foods Data And Chunking

Notebook này khám phá dữ liệu foods đã curate và cách pipeline chia nội dung thành đoạn (chunk) cho RAG. Notebook dùng lại hàm `chunk_foods_markdown()` của backend ở Phase 2, không chép lại logic chia đoạn.

Chạy từng khối mã từ repo root hoặc từ `notebooks/`; khối đầu tiên tự tìm đường dẫn `backend/` theo thư mục đang làm việc. Notebook chỉ đọc dữ liệu local, không gọi dịch vụ ngoài.

In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        backend_dir = str(base / "backend")
        if backend_dir not in sys.path:
            sys.path.insert(0, backend_dir)
        break
print("backend on sys.path")

## Khám phá dữ liệu và tổng quan chia đoạn

Hàm `chunk_foods_markdown()` quét toàn bộ curated foods Markdown, phân tích cấu trúc H1/H2, loại bỏ ảnh và mục nguồn dữ liệu, sau đó chia đoạn theo semantic sections và gắn nhãn ngữ cảnh.

In [ ]:
from collections import Counter
from statistics import median

from ingestion.chunking.markdown_chunker import chunk_foods_markdown

chunks = chunk_foods_markdown()
contents = [len(c["text"].split("\n", 1)[1]) for c in chunks]
sources = {c["metadata"]["source"] for c in chunks}

print(f"tổng số đoạn: {len(chunks)}")
print(f"số file được xử lý: {len(sources)}")
print("theo nhóm:", dict(Counter(c["metadata"]["subcategory"] for c in chunks)))
print("trường metadata:", sorted(chunks[0]["metadata"].keys()))
print(
    "độ dài phần nội dung: "
    f"trung bình {sum(contents) / len(contents):.1f}, "
    f"trung vị {median(contents)}, lớn nhất {max(contents)}"
)

## Đoạn nội dung (chunk), giới hạn 400 ký tự và nhãn ngữ cảnh

Một đoạn nội dung là đơn vị nhỏ, đứng độc lập, mà pipeline sẽ đưa vào index và retrieval. Mỗi mục H2 (`##`) tạo ra một hoặc nhiều đoạn; mục `## Nguồn dữ liệu` bị loại vì không phải nội dung trả lời.

Phần nội dung thường của mỗi đoạn giới hạn ở **400 ký tự** để tập trung đúng một ý và dễ khớp câu hỏi của người dùng.

Thứ tự ưu tiên khi chia:
1. Ranh giới đoạn văn (dòng trống).
2. Cuối câu (sau dấu chấm, chấm than, chấm hỏi).
3. Giữa các mục danh sách.
4. Khoảng trắng gần nhất trước giới hạn nếu một câu quá dài. Không bao giờ cắt giữa từ và không chồng lặp giữa hai đoạn liên tiếp.

Mỗi đoạn bắt đầu bằng một dòng nhãn `Tên tài liệu — nhãn ngắn` (ví dụ: `ANH KAFE tại Huế — địa chỉ`). Nhãn được tạo bằng quy tắc cố định và **không tính** vào giới hạn 400 ký tự.

### Ví dụ 1 — Một đoạn văn thông thường

Đoạn văn xuôi dưới 400 ký tự, có nhãn `giới thiệu`. Dòng in ra là dữ liệu đoạn thật từ pipeline; phần hiển thị Markdown bên dưới là bản xem trước trực quan.

In [ ]:
from IPython.display import Markdown, display

para = next(
    c for c in chunks
    if c["metadata"]["section"] == "Tóm tắt"
    and "|" not in c["text"].split("\n", 1)[1]
)
label_line, content = para["text"].split("\n", 1)
print("nhãn:", label_line)
print("độ dài phần nội dung:", len(content))
print("mã đoạn:", para["metadata"]["chunk_id"])
display(Markdown(para["text"]))

### Ví dụ 2 — Một bảng Markdown

Bảng Markdown dùng dấu `|` để chia cột và hàng thứ hai `---` làm đường kẻ phân cách. Nếu chia cắt giữa bảng, cấu trúc dữ liệu sẽ bị vỡ. Vì vậy bảng là **ngoại lệ có chủ ý**: giữ nguyên cả khối kể cả khi vượt 400 ký tự.

In [ ]:
table = next(c for c in chunks if "|" in c["text"].split("\n", 1)[1])
label_line, content = table["text"].split("\n", 1)
print("nhãn:", label_line)
print("độ dài phần nội dung:", len(content), "(bảng được phép vượt 400)")
print("mã đoạn:", table["metadata"]["chunk_id"])
display(Markdown(table["text"]))

### Ví dụ 3 — Một đoạn trong food-guides.md

File `food-guides.md` tổng hợp gợi ý du lịch ẩm thực (`subcategory == "guide"`). Mỗi đoạn mang nhãn là chủ đề ngắn như `ăn sáng`, `món chay` hay `tour 1 ngày`.

In [ ]:
guide = next(c for c in chunks if c["metadata"]["subcategory"] == "guide")
label_line, content = guide["text"].split("\n", 1)
print("nhãn:", label_line)
print("độ dài phần nội dung:", len(content))
print("mã đoạn:", guide["metadata"]["chunk_id"])
display(Markdown(guide["text"]))

## Các trường hợp biên đã xử lý

- Dòng chỉ chứa ảnh bị bỏ khỏi nội dung đoạn.
- Mục `## Nguồn dữ liệu` không thành đoạn; nguồn vẫn truy vết qua trường `source` trong metadata.
- Bảng Markdown luôn giữ nguyên khối, kể cả khi dài hơn 400 ký tự.
- Danh sách ưu tiên chia giữa các mục; các dòng xuống hàng thụt lề thuộc cùng mục được giữ đi cùng nhau.
- Tiêu đề phụ H3 vẫn nằm trong mục H2 của nó.
- Mã đoạn `chunk_id` là deterministic và duy nhất trên toàn corpus.